In [1]:
import pandas as pd
import numpy as np

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    make_scorer, accuracy_score, precision_score, recall_score, f1_score
)

import warnings
warnings.filterwarnings("ignore")

# ============================================================
# Settings
# ============================================================
csv_path = "../data/naffinity_dataset.csv"

CORR_METHOD = "spearman"
CORR_THRESHOLD = 0.75
N_SPLITS = 10
RANDOM_STATE = 42

# Choose which metric defines "best" hyperparameters:
PRIMARY_METRIC = "f1"  # "accuracy", "precision", "recall", "f1"

# ============================================================
# Load dataset (Train split only)
# ============================================================
df = pd.read_csv(csv_path)
train_df = df[df["Split"] == "Train"].copy()

y = train_df["Label"].astype(int)

drop_cols = {"Folder", "Split", "Label", "AffinityValue"}
feature_cols = [c for c in train_df.columns if c not in drop_cols]
X_raw = train_df[feature_cols].copy()

print(f"\nDataset: {csv_path}")
print(f"Train rows: {X_raw.shape[0]}")
print(f"Raw feature columns: {X_raw.shape[1]}")
print(f"CV: StratifiedKFold(n_splits={N_SPLITS}, shuffle=True, random_state={RANDOM_STATE})")
print(f"Fold preprocessing: NumericCleaner -> DropConstantColumns -> {CORR_METHOD}Corr(|r|>{CORR_THRESHOLD})")
print(f"Selecting best params by: mean_test_{PRIMARY_METRIC}\n")

# ============================================================
# Fold-specific preprocessing transformers
# ============================================================
class NumericCleaner(BaseEstimator, TransformerMixin):
    def __init__(self, clip_to_float32=True):
        self.clip_to_float32 = clip_to_float32

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        X = X.apply(pd.to_numeric, errors="coerce")
        X = X.replace([np.inf, -np.inf], np.nan).fillna(0.0)

        if self.clip_to_float32:
            f32max = np.finfo(np.float32).max
            X = X.clip(lower=-f32max, upper=f32max)

        return X


class DropConstantColumns(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.keep_cols_ = None

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        nunique = X.nunique(dropna=False)
        self.keep_cols_ = nunique[nunique > 1].index.tolist()
        return self

    def transform(self, X):
        X = pd.DataFrame(X)
        return X.reindex(columns=self.keep_cols_, fill_value=0.0)


class CorrelationFilter(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.8, method="spearman"):
        self.threshold = float(threshold)
        self.method = method
        self.keep_cols_ = None

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        if X.shape[1] <= 1:
            self.keep_cols_ = X.columns.tolist()
            return self

        corr = X.corr(method=self.method).abs()
        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        to_drop = [c for c in upper.columns if (upper[c] > self.threshold).any()]
        self.keep_cols_ = [c for c in X.columns if c not in set(to_drop)]
        return self

    def transform(self, X):
        X = pd.DataFrame(X)
        return X.reindex(columns=self.keep_cols_, fill_value=0.0)

# ============================================================
# Pipeline
# ============================================================
pipe = Pipeline([
    ("clean", NumericCleaner(clip_to_float32=True)),
    ("drop_const", DropConstantColumns()),
    ("corr", CorrelationFilter(threshold=CORR_THRESHOLD, method=CORR_METHOD)),
    ("model", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
])

# ============================================================
# Hyperparameter grid (as requested)
# ============================================================
param_grid = {
    "model__n_estimators": [100, 200, 300, 400, 500, 600, 700, 900, 1000],
    "model__max_depth": [
        10,
        20,
        40,
        None
    ],
    "model__min_samples_split": [
        2,
        5
    ],
    "model__min_samples_leaf": [
        1,
        2
    ],
    "model__bootstrap": [
        True,
        False
    ],
    "model__class_weight": [
        None,
        "balanced",
        "balanced_subsample"
    ]
}

# ============================================================
# Multi-metric scoring (as requested)
# ============================================================
scorers = {
    "accuracy": make_scorer(accuracy_score),
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
    "f1": make_scorer(f1_score),
}

cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring=scorers,
    refit=False,          # <-- as requested
    cv=cv,
    n_jobs=-1,
    verbose=2,
    return_train_score=False,
    error_score="raise",
)

# ============================================================
# Run grid search
# ============================================================
grid.fit(X_raw, y)

cv_results = grid.cv_results_

primary_key = f"mean_test_{PRIMARY_METRIC}"
if primary_key not in cv_results:
    raise KeyError(
        f"PRIMARY_METRIC='{PRIMARY_METRIC}' not found. "
        f"Expected '{primary_key}'. Available keys include: "
        + ", ".join([k for k in cv_results.keys() if k.startswith("mean_test_")])
    )

best_idx = int(np.argmax(cv_results[primary_key]))
best_params = cv_results["params"][best_idx]

print("\n====================")
print("🏆 Best hyperparameters")
print("====================")
print(f"Selected by highest {primary_key}: {cv_results[primary_key][best_idx]:.6f}")
print(best_params)

print("\n===============================================")
print("📊 CV metrics for best hyperparameters (mean ± std)")
print("===============================================")
print(f"Accuracy : {cv_results['mean_test_accuracy'][best_idx]:.6f} ± {cv_results['std_test_accuracy'][best_idx]:.6f}")
print(f"Precision: {cv_results['mean_test_precision'][best_idx]:.6f} ± {cv_results['std_test_precision'][best_idx]:.6f}")
print(f"Recall   : {cv_results['mean_test_recall'][best_idx]:.6f} ± {cv_results['std_test_recall'][best_idx]:.6f}")
print(f"F1       : {cv_results['mean_test_f1'][best_idx]:.6f} ± {cv_results['std_test_f1'][best_idx]:.6f}")



Dataset: ../data/naffinity_dataset.csv
Train rows: 1062
Raw feature columns: 302
CV: StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
Fold preprocessing: NumericCleaner -> DropConstantColumns -> spearmanCorr(|r|>0.75)
Selecting best params by: mean_test_f1

Fitting 10 folds for each of 864 candidates, totalling 8640 fits
[CV] END model__bootstrap=True, model__class_weight=None, model__max_depth=10, model__min_samples_leaf=1, model__min_samples_split=2, model__n_estimators=100; total time=   0.7s
[CV] END model__bootstrap=True, model__class_weight=None, model__max_depth=10, model__min_samples_leaf=1, model__min_samples_split=2, model__n_estimators=100; total time=   0.6s
[CV] END model__bootstrap=True, model__class_weight=None, model__max_depth=10, model__min_samples_leaf=1, model__min_samples_split=2, model__n_estimators=100; total time=   0.7s
[CV] END model__bootstrap=True, model__class_weight=None, model__max_depth=10, model__min_samples_leaf=1, model__min_samples_split=

KeyboardInterrupt: 